# MoSAIC-LiITA Demo

This notebook demonstrates both translation modes:
- **Deterministic**: Rule-based planner using keyword matching
- **Agentic**: LLM-powered query decomposition

**MoSAIC** — **Mo**dular **S**parql **A**ssembler for **I**nterlinked **C**orpora

## Setup

In [1]:
import json
from pathlib import Path

import requests

from mosaic_liita import (
    Planner,
    Assembler,
    QueryAgent,
    make_registry,
)
from shared.llm import create_llm_client
from shared.sparql import fetch_sparql

In [2]:
# Configuration
DATA_DIR = Path("data")
ONTOLOGY_PATH = DATA_DIR / "ontology_filtered.json"
LIITA_ENDPOINT = "https://liita.it/sparql"

In [3]:
# Load catalog and initialize components
with open(ONTOLOGY_PATH, "r", encoding="utf-8") as f:
    catalog = json.load(f)["documents"]

registry = make_registry()
planner = Planner(registry, catalog)
assembler = Assembler()

print(f"Loaded {len(catalog)} catalog entries")
print(f"Registry contains {len(registry.blocks)} blocks")

Loaded 186 catalog entries
Registry contains 20 blocks


In [4]:
registry.blocks['TRANSLATE_TO_SICILIANO']

Block(id='TRANSLATE_TO_SICILIANO', requires={'?liitaLemma'}, provides={'?sicWR', '?sicLemma'}, prefixes={'ontolex', 'vartrans', 'dcterms'}, where=['?itToSicEntry ontolex:canonicalForm ?liitaLemma ;', '             vartrans:translatableAs ?sicEntry .', '?sicEntry ontolex:canonicalForm ?sicLemma .', '?sicLemma dcterms:isPartOf <http://liita.it/data/id/DialettoSiciliano/lemma/LemmaBank> ;', '         ontolex:writtenRep ?sicWR .'], service_iri=None)

In [5]:
def execute_sparql(sparql: str, limit: int = 10) -> dict:
    """Execute a SPARQL query against the LiITA endpoint."""
    return fetch_sparql(sparql, limit=limit)


def display_results(data: dict):
    """Display SPARQL results in a readable format."""
    variables = data.get("head", {}).get("vars", [])
    bindings = data.get("results", {}).get("bindings", [])

    print(f"Results: {len(bindings)} rows")
    print(f"Variables: {', '.join(variables)}\n")

    for i, row in enumerate(bindings, 1):
        print(f"--- Row {i} ---")
        for var in variables:
            if var in row:
                val = row[var].get("value", "")
                print(f"  {var}: {val}")
        print()

---

## Deterministic Mode

The deterministic planner uses rule-based pattern matching to select blocks. No LLM required.

In [6]:
def translate_deterministic(question: str) -> str:
    """Translate a natural language question to SPARQL using the deterministic planner."""
    spec = planner.plan(question)
    plan = spec.compile(registry)
    sparql = assembler.assemble(plan)
    
    # Show plan details
    print("=" * 60)
    print(f"Query: {question}")
    print("=" * 60)
    print(f"\nBlocks used ({len(plan.blocks)}):")
    for i, bi in enumerate(plan.blocks, 1):
        print(f"  {i}. {bi.block.id}")
        if bi.slots:
            for k, v in bi.slots.items():
                if v:
                    print(f"     - {k}: {v}")
    
    print(f"\nSelect variables: {plan.select_vars}")
    if plan.aggregates:
        print(f"Aggregates: {plan.aggregates}")
    
    print("\n" + "-" * 60)
    print("Generated SPARQL:")
    print("-" * 60)
    print(sparql)
    
    return sparql

### Example 1: Find synonyms of a word

In [7]:
sparql = translate_deterministic("Find synonyms of 'origine'")

Query: Find synonyms of 'origine'

Blocks used (1):
  1. COMPLIT_SEMREL_OF_SEED_LEMMA
     - seed_lemma: "origine"
     - rel_triple: { ?seedSense lexinfo:approximateSynonym ?senseRel . } UNION { ?senseRel lexinfo:approximateSynonym ?seedSense . }

Select variables: ['?wordRel']
Aggregates: {'?definitionSample': 'SAMPLE(?definition)'}

------------------------------------------------------------
Generated SPARQL:
------------------------------------------------------------
PREFIX lexinfo: <http://www.lexinfo.net/ontology/3.0/lexinfo#>
PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?wordRel (SAMPLE(?definition) AS ?definitionSample)
WHERE {
  SERVICE <https://klab.ilc.cnr.it/graphdb-compl-it/> {
    ?seedWord a ontolex:Word ;
              lexinfo:partOfSpeech [ rdfs:label ?posLabel ] ;
              ontolex:canonicalForm [ ontolex:writtenRep ?seedForm ] ;
            

In [8]:
# Execute the query
results = execute_sparql(sparql)
display_results(results)

Results: 7 rows
Variables: wordRel, definitionSample

--- Row 1 ---
  wordRel: http://lexica/mylexicon#MUSgenesiNOUN
  definitionSample: origine, formazione

--- Row 2 ---
  wordRel: http://lexica/mylexicon#MUSprincipioNOUN
  definitionSample: inizio

--- Row 3 ---
  wordRel: http://lexica/mylexicon#MUSgermeNOUN
  definitionSample: origine, causa prima

--- Row 4 ---
  wordRel: http://lexica/mylexicon#MUSfonteNOUN2
  definitionSample: persona o cosa da cui proviene qualcosa

--- Row 5 ---
  wordRel: http://lexica/mylexicon#MUSmadreNOUN
  definitionSample: principio origine, con riferimento a una maternita' figurata

--- Row 6 ---
  wordRel: http://lexica/mylexicon#MUSfiliazioneNOUN
  definitionSample: provenienza, origine

--- Row 7 ---
  wordRel: http://lexica/mylexicon#MUScausaNOUN
  definitionSample: cio' che e' all'origine di qualcosa; cio' che produce un effetto; motivo, ragione



### Example 2: Find definitions by pattern

In [9]:
sparql = translate_deterministic("Find definitions of words starting with 'ante'")

Query: Find definitions of words starting with 'ante'

Blocks used (1):
  1. COMPLIT_DEF_FILTER_BY_PATTERN
     - lemma_filter: FILTER(STRSTARTS(STR(?itLemmaString), "ante")) .

Select variables: ['?itLemmaString', '?definition']

------------------------------------------------------------
Generated SPARQL:
------------------------------------------------------------
PREFIX lexinfo: <http://www.lexinfo.net/ontology/3.0/lexinfo#>
PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?itLemmaString ?definition
WHERE {
  SERVICE <https://klab.ilc.cnr.it/graphdb-compl-it/> {
    ?word a ontolex:Word ;
          lexinfo:partOfSpeech [ rdfs:label ?posLabel ] ;
          ontolex:sense ?sense ;
          ontolex:canonicalForm ?form .
    ?form ontolex:writtenRep ?itLemmaString .
    FILTER(STRSTARTS(STR(?itLemmaString), "ante")) .
    OPTIONAL { ?sense skos:definition ?definition .

In [10]:
results = execute_sparql(sparql)
display_results(results)

Results: 36 rows
Variables: itLemmaString, definition

--- Row 1 ---
  itLemmaString: antecedente
  definition: evento che ne precede un altro

--- Row 2 ---
  itLemmaString: antecedente
  definition: conoscere gli antecedenti

--- Row 3 ---
  itLemmaString: antecedenza
  definition: Qualità di ciò che è antecedente

--- Row 4 ---
  itLemmaString: antecessore
  definition: Chi ha preceduto qualcuno

--- Row 5 ---
  itLemmaString: antefatto
  definition: avvenimento che precede un fatto

--- Row 6 ---
  itLemmaString: antefatto
  definition: raccontare l'antefatto

--- Row 7 ---
  itLemmaString: antefissa
  definition: Elemento a funzione prevalentemente decorativa

--- Row 8 ---
  itLemmaString: anteguerra
  definition: periodo che precede una guerra

--- Row 9 ---
  itLemmaString: anteguerra
  definition: un episodio che risale all'anteguerra

--- Row 10 ---
  itLemmaString: antemurale
  definition: Costruzione avanzata nelle fortificazioni ellenistiche, romane e medievali

--- Row 11

### Example 3: Translation to dialect

In [11]:
sparql = translate_deterministic("Find Sicilian lemmas ending with 'u' and translate them into Italian")

Query: Find Sicilian lemmas ending with 'u' and translate them into Italian

Blocks used (3):
  1. SICILIANO_LEMMA_FILTER_BY_PATTERN_AND_POS
     - wr_filter: FILTER(REGEX(STR(?sicWR), "u$")) .
  2. TRANSLATE_FROM_SICILIANO
  3. ASSERT_LIITA_LEMMA_TYPE

Select variables: ['?liitaLemma', '?itLemmaString', '?sicLemma', '?sicWR']

------------------------------------------------------------
Generated SPARQL:
------------------------------------------------------------
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX lila: <http://lila-erc.eu/ontologies/lila/>
PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
PREFIX vartrans: <http://www.w3.org/ns/lemon/vartrans#>

SELECT ?liitaLemma ?itLemmaString ?sicLemma ?sicWR
WHERE {
    ?sicLemma a lila:Lemma .
    ?sicLemma dcterms:isPartOf <http://liita.it/data/id/DialettoSiciliano/lemma/LemmaBank> .
    
    ?sicLemma ontolex:writtenRep ?sicWR .
    FILTER(REGEX(STR(?sicWR), "u$")) .
  ?sicEntry ontolex:canonicalForm ?sicLemma .
  ?itEntry 

In [12]:
results = execute_sparql(sparql)
display_results(results)

Results: 200 rows
Variables: liitaLemma, itLemmaString, sicLemma, sicWR

--- Row 1 ---
  liitaLemma: http://liita.it/data/id/lemma/959836
  itLemmaString: abbacchiamento
  sicLemma: http://liita.it/data/id/DialettoSiciliano/lemma/12152
  sicWR: abbattimentu

--- Row 2 ---
  liitaLemma: http://liita.it/data/id/lemma/959847
  itLemmaString: abbagliamento
  sicLemma: http://liita.it/data/id/DialettoSiciliano/lemma/12148
  sicWR: abbagghiamentu

--- Row 3 ---
  liitaLemma: http://liita.it/data/id/lemma/959847
  itLemmaString: abbagliamento
  sicLemma: http://liita.it/data/id/DialettoSiciliano/lemma/12148
  sicWR: abbagliamentu

--- Row 4 ---
  liitaLemma: http://liita.it/data/id/lemma/959850
  itLemmaString: abbaglio
  sicLemma: http://liita.it/data/id/DialettoSiciliano/lemma/13507
  sicWR: abbagghiu

--- Row 5 ---
  liitaLemma: http://liita.it/data/id/lemma/959850
  itLemmaString: abbaglio
  sicLemma: http://liita.it/data/id/DialettoSiciliano/lemma/13507
  sicWR: abbagliu

--- Row 6 ---
 

### Example 4: Emotions and semantic relations

In [13]:
sparql = translate_deterministic(
    "Find Italian words that express positive emotions ('gioia') and are hyponyms of 'sentimento'"
)

Query: Find Italian words that express positive emotions ('gioia') and are hyponyms of 'sentimento'

Blocks used (5):
  1. COMPLIT_SEMREL_OF_SEED_LEMMA
     - seed_lemma: "sentimento"
     - rel_triple: ?senseRel lexinfo:hyponym ?seedSense .
  2. JOIN_WORDREL_TO_LIITA
  3. ASSERT_LIITA_LEMMA_TYPE
  4. ELITA_EMOTION_FILTER
     - emotion_filter_clause: VALUES ?emotion { elita:Gioia }
  5. SENTIX_POLARITY

Select variables: ['?liitaLemma', '?itLemmaString']
Aggregates: {'?definitionSample': 'SAMPLE(?definition)', '?emotions': 'GROUP_CONCAT(DISTINCT ?emotionLabel; SEPARATOR=", ")', '?polarityLabels': 'GROUP_CONCAT(DISTINCT ?polarityLabel; SEPARATOR=", ")', '?polarityValues': 'GROUP_CONCAT(DISTINCT ?polarityValue; SEPARATOR=", ")'}

------------------------------------------------------------
Generated SPARQL:
------------------------------------------------------------
PREFIX elita: <http://w3id.org/elita/>
PREFIX lexinfo: <http://www.lexinfo.net/ontology/3.0/lexinfo#>
PREFIX lila: <http:

In [14]:
results = execute_sparql(sparql)
display_results(results)

Results: 16 rows
Variables: liitaLemma, itLemmaString, definitionSample, emotions, polarityLabels, polarityValues

--- Row 1 ---
  liitaLemma: http://liita.it/data/id/lemma/962798
  itLemmaString: allegria
  definitionSample: stato d'animo
  emotions: Gioia
  polarityLabels: 
  polarityValues: 0.4352000057697296, 0.5

--- Row 2 ---
  liitaLemma: http://liita.it/data/id/lemma/963388
  itLemmaString: amicizia
  definitionSample: legame tra persone basato su affetto stima
  emotions: Gioia
  polarityLabels: 
  polarityValues: 0.09000000357627869, 0.625

--- Row 3 ---
  liitaLemma: http://liita.it/data/id/lemma/963681
  itLemmaString: amore
  definitionSample: affetto reciproco
  emotions: Gioia
  polarityLabels: 
  polarityValues: 0.4580000042915344, 0.53329998254776

--- Row 4 ---
  liitaLemma: http://liita.it/data/id/lemma/990222
  itLemmaString: esultanza
  definitionSample: Manifestazione di gioia
  emotions: Gioia
  polarityLabels: 
  polarityValues: -0.2290000021457672, 0.5830000042

---

## Agentic Mode

The agentic translator uses an LLM to decompose queries into tool operations.

**Requirements:** Configure your LLM provider and API key below.

In [ ]:
# LLM Configuration
# Choose your provider: "mistral", "anthropic", "openai", "gemini", "ollama"

LLM_PROVIDER = "mistral"  # Change this to your provider
LLM_API_KEY = ""         # Not needed for Ollama
LLM_MODEL = ""   # Or leave empty for default

# Default models per provider:
# - mistral: "mistral-large-latest"
# - anthropic: "claude-sonnet-4-20250514"
# - openai: "gpt-4.1-mini"
# - gemini: "gemini-1.5-pro"
# - ollama: "llama3.1"

In [ ]:
# Initialize LLM client and agent
llm_client = create_llm_client(
    provider=LLM_PROVIDER,
    api_key=LLM_API_KEY if LLM_API_KEY else None,
    model=LLM_MODEL,
)

agent = QueryAgent(registry, catalog, llm_client)
print(f"Agent initialized with {LLM_PROVIDER} ({LLM_MODEL or 'default model'})")

In [ ]:
def translate_agentic(question: str) -> str:
    """Translate a natural language question to SPARQL using the agentic planner."""
    sparql, agent_plan, query_spec = agent.translate(question)
    
    print("=" * 60)
    print(f"Query: {question}")
    print("=" * 60)
    
    print(f"\nAgent Reasoning:")
    print(f"  {agent_plan.reasoning}")
    
    print(f"\nTool Calls ({len(agent_plan.steps)}):")
    for i, step in enumerate(agent_plan.steps, 1):
        print(f"  {i}. {step.tool}")
        if step.params:
            for k, v in step.params.items():
                print(f"     - {k}: {v}")
    
    print(f"\nOutput variables: {agent_plan.output_vars}")
    
    if agent_plan.aggregation:
        print(f"Aggregation: {agent_plan.aggregation}")
    
    print("\n" + "-" * 60)
    print("Generated SPARQL:")
    print("-" * 60)
    print(sparql)
    
    return sparql

### Example 1: Complex query with filtering

In [ ]:
sparql = translate_agentic("Find synonyms of 'origine' whose Sicilian translation starts with 'm'")

In [ ]:
results = execute_sparql(sparql)
display_results(results)

### Example 2: Counting results

In [ ]:
sparql = translate_agentic("How many lemmas are present in the Sicilian lexicon?")

In [ ]:
results = execute_sparql(sparql)
display_results(results)

### Example 3: Multi-step translation

In [ ]:
sparql = translate_agentic(
    "Find CompL-IT definitions starting with 'uccello' and show Parmigiano and Sicilian translations"
)

In [ ]:
results = execute_sparql(sparql)
display_results(results)

---

## Comparison

Try the same query with both modes to see the differences.

In [ ]:
test_query = "Find synonyms of 'origine' with their Sicilian translations"

print("DETERMINISTIC MODE")
print("=" * 60)
sparql_det = translate_deterministic(test_query)

In [ ]:
print("\nAGENTIC MODE")
print("=" * 60)
sparql_agent = translate_agentic(test_query)

In [ ]:
# Compare results
print("Deterministic results:")
display_results(execute_sparql(sparql_det))

print("\nAgentic results:")
display_results(execute_sparql(sparql_agent))